# Instalación de librerías y comprobación de entorno


In [1]:
# Dependencias necesarias — instalar antes de ejecutar este notebook:
#
#   GPU NVIDIA (CUDA 12.4 — RTX 4070 Super / driver >= 525):
#     pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
#
#   Resto del stack:
#     pip install -r requirements/finetuning.txt

import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}", end="")
if torch.cuda.is_available():
    print(f" | GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB)")
else:
    print("\nSin GPU: el entrenamiento será muy lento. Se recomienda GPU con >= 8 GB VRAM.")

PyTorch 2.6.0+cu124 | CUDA: True | GPU: NVIDIA GeForce RTX 4070 SUPER (13 GB)


In [2]:
import os
import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path
import sys

# Asegura que el directorio de trabajo sea la raíz del repositorio
# (necesario si Jupyter se abrió desde src/finetuning/ en vez de desde la raíz)
_here = Path.cwd()
if not (_here / "requirements").exists():
    for _parent in _here.parents:
        if (_parent / "requirements").exists():
            os.chdir(_parent)
            break
    else:
        raise RuntimeError(
            f"No se encontró la raíz del repositorio desde {_here}.\n"
            "Ejecuta Jupyter desde la raíz del proyecto: jupyter lab"
        )

# Importar funciones auxiliares del módulo local
sys.path.insert(0, str(Path.cwd() / "src" / "finetuning"))
from functions import (
    LABEL_RELEVANTE, LABEL_NO_RELEVANTE, LABELS, GRADING_SYSTEM_PROMPT,
    check_gpu, load_grading_dataset, split_dataset, show_split_stats,
    build_grading_messages, format_training_prompt, examples_to_hf_dataset,
    load_tokenizer, load_model_4bit, build_peft_model, get_training_args,
    run_training, save_adapter, predict_relevance, evaluate_model,
    print_comparison, log_experiment,
)

check_gpu()

PyTorch:          2.6.0+cu124
CUDA disponible:  True
GPU:              NVIDIA GeForce RTX 4070 SUPER
VRAM total:       12.9 GB


In [ ]:
from pathlib import Path

# El CWD ya fue fijado a la raíz del repositorio en la celda anterior (env-check-05)

# Rutas locales (relativas a la raíz del repositorio)
DATASET_PATH = Path("data/processed/grading_dataset.jsonl")

LABEL_RELEVANTE    = "relevante"
LABEL_NO_RELEVANTE = "no relevante"
LABELS = [LABEL_RELEVANTE, LABEL_NO_RELEVANTE]

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

OUTPUT_DIR   = "src/finetuning/output/qwen-grader-lora"
ADAPTER_PATH = "src/finetuning/output/qwen-grader-lora/adapter_final"
MERGED_PATH  = "src/finetuning/output/qwen-grader-merged"

MAX_SEQ_LEN = 512

MLFLOW_EXPERIMENT_NAME = "grader_relevancia_qwen25_3b"

print(f"Directorio de trabajo: {Path.cwd()}")
print("Configuración:")
print(f"  Modelo base:    {MODEL_ID}")
print(f"  Dataset:        {DATASET_PATH}  (existe: {DATASET_PATH.exists()})")
print(f"  Output dir:     {OUTPUT_DIR}")
print(f"  Adapter path:   {ADAPTER_PATH}")
print(f"  Max seq len:    {MAX_SEQ_LEN}")
print(f"  MLflow exp:     {MLFLOW_EXPERIMENT_NAME}")

In [ ]:
# ── Valores del entrenamiento (NB02 — celdas lora-config-20 y trainer-run-22) ─
train_loss       = 0.7306      # train_result.training_loss
train_runtime    = 1435.0      # train_result.metrics['train_runtime'] (segundos)
trainable_params = 14_966_784  # model.print_trainable_parameters() → 0.48% de 3.1B

# ── Splits del dataset ────────────────────────────────────────────────────────
train_size = 443
val_size   = 95
test_size  = 96

# ── Métricas baseline (NB03 — celda baseline-eval-14) ────────────────────────
# Test set: 96 ejemplos | 43 relevante / 53 no relevante
baseline_acc = 0.8542
baseline_f1  = 0.8500
baseline_precision_rel   = 0.89
baseline_recall_rel      = 0.77
baseline_precision_norel = 0.83
baseline_recall_norel    = 0.92

# ── Métricas fine-tuned (NB03 — celda eval-ft-24) ────────────────────────────
# Test set: 96 ejemplos | 43 relevante / 53 no relevante
finetuned_acc = 0.9062
finetuned_f1  = 0.9057
finetuned_precision_rel   = 0.87
finetuned_recall_rel      = 0.93
finetuned_precision_norel = 0.94
finetuned_recall_norel    = 0.89

print(f"baseline_f1   = {baseline_f1:.4f}")
print(f"finetuned_f1  = {finetuned_f1:.4f}")
print(f"mejora F1     = {finetuned_f1 - baseline_f1:+.4f}")
print(f"train_loss    = {train_loss:.4f}  |  runtime = {train_runtime:.0f}s  |  params entrenables = {trainable_params:,}")

# Seguimiento con MLflow

Registramos los parámetros y métricas en el servidor MLflow del proyecto
(configurado via `MLFLOW_TRACKING_URI` en `.env`), siguiendo el mismo patrón
que el clasificador de riesgo (`src/classifier/`).


In [ ]:
import mlflow
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

tracking_uri = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

mejora_f1  = finetuned_f1  - baseline_f1
mejora_acc = finetuned_acc - baseline_acc

try:
    with mlflow.start_run(run_name="qwen25-3b-qlora-grader") as run:

        # ── Parámetros del experimento ────────────────────────────────────────
        mlflow.log_params({
            "model_id":            MODEL_ID,
            "max_seq_len":         MAX_SEQ_LEN,
            "lora_r":              8,
            "lora_alpha":          16,
            "lora_dropout":        0.05,
            "target_modules":      "q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj",
            "epochs":              3,
            "learning_rate":       2e-4,
            "batch_size_efectivo": 16,
            "optimizer":           "paged_adamw_8bit",
            "lr_scheduler":        "cosine",
            "quantization":        "4bit_nf4",
            "trainable_params":    trainable_params,
            "train_size":          train_size,
            "val_size":            val_size,
            "test_size":           test_size,
        })

        # ── Métricas ─────────────────────────────────────────────────────────
        mlflow.log_metrics({
            # Globales
            "baseline_accuracy":               baseline_acc,
            "baseline_f1_macro":               baseline_f1,
            "finetuned_accuracy":              finetuned_acc,
            "finetuned_f1_macro":              finetuned_f1,
            "mejora_accuracy":                 mejora_acc,
            "mejora_f1_macro":                 mejora_f1,
            # Por clase — baseline
            "baseline_precision_relevante":    baseline_precision_rel,
            "baseline_recall_relevante":       baseline_recall_rel,
            "baseline_precision_no_relevante": baseline_precision_norel,
            "baseline_recall_no_relevante":    baseline_recall_norel,
            # Por clase — fine-tuned
            "finetuned_precision_relevante":    finetuned_precision_rel,
            "finetuned_recall_relevante":       finetuned_recall_rel,
            "finetuned_precision_no_relevante": finetuned_precision_norel,
            "finetuned_recall_no_relevante":    finetuned_recall_norel,
            # Entrenamiento
            "train_loss":       train_loss,
            "train_runtime_s":  train_runtime,
        })

        # ── Dataset: artefacto + lineage tracking ────────────────────────────
        if DATASET_PATH.exists():
            mlflow.log_artifact(str(DATASET_PATH), artifact_path="dataset")
            df_all = pd.read_json(DATASET_PATH, lines=True)
            mlflow_dataset = mlflow.data.from_pandas(
                df_all,
                source=str(DATASET_PATH),
                name="grading_dataset",
                targets="label",
            )
            mlflow.log_input(mlflow_dataset, context="training")
            print(f"  Dataset subido: {len(df_all)} ejemplos → artifacts/dataset/")
        else:
            print(f"  AVISO: dataset no encontrado en {DATASET_PATH}")

        # ── Metadatos del adaptador LoRA ──────────────────────────────────────
        adapter_meta = Path(ADAPTER_PATH) / "model_metadata.json"
        if adapter_meta.exists():
            mlflow.log_artifact(str(adapter_meta), artifact_path="model")
            print(f"  Metadatos del adaptador subidos → artifacts/model/")

        print(f"\nExperimento registrado correctamente.")
        print(f"  Experimento: {MLFLOW_EXPERIMENT_NAME}")
        print(f"  Run ID:      {run.info.run_id}")
        print(f"  URI:         {tracking_uri}")
        print(f"  Mejora F1:   {mejora_f1:+.4f}  |  Mejora Acc: {mejora_acc:+.4f}")

except Exception as e:
    print(f"MLflow no disponible: {e}")
    print(f"  baseline_f1={baseline_f1:.4f}  finetuned_f1={finetuned_f1:.4f}  mejora={mejora_f1:+.4f}")

# Conclusiones

## Resultados

| Modelo | Accuracy | F1-macro |
|--------|:--------:|:--------:|
| Qwen 2.5 3B-Instruct (baseline, prompting) | 0.8542 | 0.8500 |
| Qwen 2.5 3B-Instruct + QLoRA (fine-tuned) | **0.9062** | **0.9057** |
| Mejora | +0.0521 | **+0.0557** |

### Por clase

| Modelo | Clase | Precision | Recall | F1 |
|--------|-------|:---------:|:------:|:--:|
| Baseline | relevante | 0.89 | 0.77 | 0.82 |
| Baseline | no relevante | 0.83 | 0.92 | 0.88 |
| Fine-tuned | relevante | 0.87 | **0.93** | **0.90** |
| Fine-tuned | no relevante | **0.94** | 0.89 | **0.91** |

El fine-tuning mejora sobre todo el **recall de "relevante"** (+0.16) y la **precision de "no relevante"** (+0.11).

### Entrenamiento

- **Modelo base:** Qwen 2.5 3B-Instruct | Cuantización: 4-bit NF4 (QLoRA)
- **Parámetros entrenables:** 14,966,784 (0.48% de 3.1B)
- **Loss final:** 0.7306 | **Tiempo:** 1435 s (~24 min en RTX 4070 Super)
- **Épocas:** 3 | **Batch efectivo:** 16 | **LR:** 2e-4 (cosine)

## Dataset utilizado

- **634 ejemplos** · 117 queries sobre EU AI Act, RGPD y AESIA
- Split: 70% train (443) / 15% val (95) / 15% test (96)
- Balanceo: 44.6% relevantes / 55.4% no relevantes
- Negativos: hard negatives del mismo dominio legal + easy negatives de leyes ajenas
- Generado con `data/generate_grading_dataset.py`

## Limitaciones

- **Tamaño del dataset**: 634 ejemplos es suficiente para un primer experimento pero modesto.
  Ampliar con más queries o con datos reales del retriever ChromaDB mejorará la generalización.
- **Datos sintéticos**: los fragmentos documentales del dataset se generaron a partir de las fuentes
  reales (EU AI Act, RGPD, AESIA) pero no son los chunks exactos de ChromaDB.
  Idealmente el dataset final usaría los chunks reales que devuelve el retriever en producción.
- **Integración con Ollama**: el adaptador LoRA requiere cargar el modelo base en memoria.
  Para producción, hacer merge y convertir a GGUF.

## Próximos pasos

1. **Integrar** en `src/rag/main.py` con la función `grade_with_finetuned()` (fragmento en NB03)
2. **Ampliar el dataset** con chunks reales del retriever ChromaDB
3. *(Opcional)* Merge + conversión a GGUF + push a Ollama para producción

## Impacto en el pipeline

Un grader especializado reduce el ruido en el contexto de generación:
- Menos documentos irrelevantes → respuestas más precisas y menos alucinaciones en citas legales
- Menor coste de tokens en la llamada a Bedrock Nova Lite (contexto más corto)

---
_Informe preliminar generado por IA. Consulte profesional jurídico._